# Codec de comma (VQ-VAE)
`python -m src.cli codec <acción>` en un notebook: cada sección es una acción de la CLI, en el mismo orden y con los mismos defaults, más una sección propia del modelo. El encoder y decoder del VQ-VAE de comma que convierte cámara en 128 tokens por frame y viceversa. No se entrena acá: se mide (fidelidad fp32 contra fp16 y latencia por dispositivo) y se exporta a ONNX.

Guía completa: `Docs/modelos/README.md`. Acciones de `codec`: `bench`, `exportar`.

**Cómo se usa.** Con `DRY_RUN = True` (lo de la primera celda) cada acción solo imprime los comandos que correría, igual que `--dry-run`. Con `DRY_RUN = False` corre de verdad por `Toolkit`, en segundo plano (el kernel sigue libre), con candado de GPU y cola, y queda en `results/bitacora.jsonl` con su línea de CLI exacta. Lo que todavía no existe (resultados, checkpoints) se muestra como *pendiente*.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

In [ ]:
MODELO = "codec"
RUN = "codec"       # nombre de la corrida: <modelo>_<tamaño>[_<variante>][_s<semilla>]
DRY_RUN = True      # True: solo imprime los comandos (= --dry-run). False: los corre de verdad, en segundo plano
DEVICE = nbui.device_inicial()      # el que detecta el repo: cuda, rocm, mps, cpu (dml y npu: con ONNX Runtime)
DEV = nbui.device_kw(DEVICE)        # = --device
tk = Toolkit(MODELO)
print(MODELO, "| acciones:", ", ".join(tk.acciones), "| corrida:", RUN, "| device:", DEVICE, "| dry-run:", DRY_RUN)

## 1. Receta y fidelidad
La receta de entrenamiento es la de la fuente del modelo (con lo forzado por datos o presupuesto marcado) y la tabla de fidelidad dice qué es fiel, qué lo fuerzan los datos o el presupuesto y qué es propio. CLI: `python -m src.cli codec receta`.

In [ ]:
nbui.fidelidad(MODELO);

`codec` no tiene receta de entrenamiento: no se entrena en el repo.

## 2. Entrenar o continuar
Entrenar de cero con la receta de la fuente, probar el lr en un humo corto, seguir la corrida en vivo y continuarla o extenderla. CLI: `entrenar` (con `--humo`), `continuar`, `estado`.

No aplica: `codec` no se entrena en el repo.

## 3. Evaluar
Métricas propias de video: CE, rollouts de 3 s, LPIPS y FVD con IC por segmento, y el piso de FVD (real contra real) al lado. CLI: `python -m src.cli codec evaluar`.

No aplica: `codec` no declara la acción `evaluar`.

## 4. Latencia
Un protocolo único: un paso por frame en batch 1, fp16 con CUDA graphs donde anda, p50 y p99; con `--cadena` mide la cadena cámara → encoder → modelo → decoder, en bucle cerrado y a ritmo de cámara (50 ms). CLI: `python -m src.cli codec bench`.

In [ ]:
OPC_BENCH = dict(limite=LIMITE, **DEV)
nbui.opciones(MODELO, "bench", **OPC_BENCH)
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, **OPC_BENCH)        # latencia del codec y fidelidad fp32 contra fp16

In [ ]:
nbui.latencia_etapas();      # p50 y p99 de cada columna del protocolo y mediana por etapa, de results/infer/

## 5. Demo
Un mp4 con el video real a la izquierda y lo que predice el modelo a la derecha: 1 s de contexto real y después imagina. CLI: `python -m src.cli codec demo`.

No aplica: `codec` no declara la acción `demo`.

## 6. Exportar
ONNX, donde aplica (`python -m src.cli codec exportar`). El export verifica que ONNX Runtime coincida con PyTorch.

In [ ]:
OPC_EXPORTAR = dict(run=RUN) if MODELO != "codec" else dict()
nbui.opciones(MODELO, "exportar", **OPC_EXPORTAR)
t = nbui.lanzar(MODELO, "exportar", dry_run=DRY_RUN, **OPC_EXPORTAR)
t = nbui.lanzar(MODELO, "exportar", dry_run=DRY_RUN, dtype="fp16", **OPC_EXPORTAR)      # la versión fp16

## 7. Sección propia: fidelidad y latencia
El decoder en fp16 tiene que dar los mismos tokens y casi la misma imagen que en fp32; la latencia se mide por dispositivo (cuda, cpu, rocm, mps, dml, npu) con el mismo protocolo.

In [ ]:
nbui.backends();

In [ ]:
t = nbui.lanzar("codec", "bench", dry_run=DRY_RUN, limite=LIMITE, **DEV)
nbui.codec_resultados();